# Optimizer Comparison on Real Data (Red Wine Quality)

## Introduction
In this notebook, we will apply various optimization algorithms to a real-world dataset: **Red Wine Quality**.

This dataset contains 1599 samples with 11 physicochemical properties (features) and a quality score (target).

We will treat this as a **Regression** problem (predicting the quality score).

**Objective**:
- Compare the convergence speed (Loss vs Epochs) of different optimizers.
- Compare the computational time required for training.
- Observe the effect of **Regularization** (Weight Decay, Dropout, BatchNorm) and **Activation Functions**.

**Optimizers to Test**:
1.  **SGD (Stochastic Gradient Descent)**
2.  **SGD with Momentum**
3.  **Nesterov Accelerated Gradient (NAG)**
4.  **Adagrad**
5.  **RMSprop**
6.  **Adam**

**Concepts Covered**:
- **Nesterov Momentum**: A lookahead momentum that corrects the update direction.
- **Adagrad**: Adapts learning rates for each parameter (good for sparse data).
- **Weight Decay (L2 Regularization)**: Adds a penalty term to the loss to prevent overfitting.
- **Dropout**: Randomly zeros out neurons during training to improve generalization.
- **Batch Normalization**: Normalizes layer inputs to have zero mean and unit variance. This stabilizes training, allows for higher learning rates, and accelerates convergence by reducing internal covariate shift.
- **LeakyReLU / PReLU / ReLU6**: Variants of ReLU to handle specific issues like dead neurons or capping activations.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Data Loading and Preprocessing

We load the dataset directly from the UCI Machine Learning Repository.

In [ ]:
# Load dataset
# url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
url = "data/winequality-red.csv"
print(f"Downloading data from {url}...")
df = pd.read_csv(url, sep=';')

print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Preprocessing
X = df.drop('quality', axis=1).values
y = df['quality'].values.reshape(-1, 1)

## Visualizations Correlation matrix and Quality Distribution

In [ ]:
import seaborn as sns

# Correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Feature Correlation Matrix")
plt.tight_layout()

# Quality Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='quality', data=df)
plt.title("Wine Quality Distribution")
plt.tight_layout()

## Standardize features (Crucial for Neural Networks!)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to PyTorch tensors
X_tensor = torch.FloatTensor(X_scaled)
y_tensor = torch.FloatTensor(y)

## Train Test Split

In [ ]:
# Split into Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# Create DataLoaders
batch_size = 64
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

## 2. Model Definition

We use a simple Feedforward Neural Network with 2 hidden layers.

64 neurons in first hidden layer and 
32 neurons in second hidden layer.

In [ ]:
class WineQualityNet(nn.Module):
    def __init__(self, input_dim, activation='relu', dropout_prob=0.0, use_batchnorm=False):
        super(WineQualityNet, self).__init__()
        self.use_batchnorm = use_batchnorm
        
        self.fc1 = nn.Linear(input_dim, 64)
        if use_batchnorm:
            self.bn1 = nn.BatchNorm1d(64)
            
        self.fc2 = nn.Linear(64, 32)
        if use_batchnorm:
            self.bn2 = nn.BatchNorm1d(32)
            
        self.fc3 = nn.Linear(32, 1) # Output layer (Regression)
        
        self.dropout = nn.Dropout(p=dropout_prob)
        
        # Select activation function
        if activation == 'relu':
            self.act = nn.ReLU()
        elif activation == 'leaky_relu':
            self.act = nn.LeakyReLU(0.1)
        elif activation == 'prelu':
            self.act = nn.PReLU()
        elif activation == 'relu6':
            self.act = nn.ReLU6()
        elif activation == 'tanh':
            self.act = nn.Tanh()
        elif activation == 'sigmoid':
            self.act = nn.Sigmoid()
        else:
            self.act = nn.ReLU()
        
    def forward(self, x):
        out = self.fc1(x)
        if self.use_batchnorm:
            out = self.bn1(out)
        out = self.act(out)
        out = self.dropout(out)
        
        out = self.fc2(out)
        if self.use_batchnorm:
            out = self.bn2(out)
        out = self.act(out)
        out = self.dropout(out)
        
        out = self.fc3(out)
        return out

input_dim = X_train.shape[1]

In [ ]:
print(f"X_train.shape: {X_train.shape}\n {X_train}")

## 3. Training Helper Function

This function trains the model using a specific optimizer and returns the loss history and training time.

In [ ]:
def train_model(optimizer_name, learning_rate=0.001, epochs=100, weight_decay=0.0, activation='relu', dropout=0.0, use_batchnorm=False):
    # Initialize model with specific configuration
    model = WineQualityNet(input_dim, activation=activation, dropout_prob=dropout, use_batchnorm=use_batchnorm)
    criterion = nn.MSELoss()
    
    # Optimizer Selection
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'SGD_Momentum':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay)
    elif optimizer_name == 'Nesterov':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, nesterov=True, weight_decay=weight_decay)
    elif optimizer_name == 'Adagrad':
        optimizer = optim.Adagrad(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    loss_history = []
    start_time = time.time()
    
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        loss_history.append(epoch_loss / len(train_loader))
        
    end_time = time.time()
    duration = end_time - start_time
    
    return loss_history, duration

## 4. Running Experiments

We will run the training loop for each optimizer with different configurations and collect the results.

In [ ]:
# Experiment Configurations
configs = [
    # --- Baselines ---
    {'name': 'SGD_ReLU', 'opt': 'SGD', 'lr': 0.001, 'wd': 0.0, 'act': 'relu', 'drop': 0.0, 'bn': False},
    {'name': 'SGD_Momentum', 'opt': 'SGD_Momentum', 'lr': 0.001, 'wd': 0.0, 'act': 'relu', 'drop': 0.0, 'bn': False},
    
    # --- Adaptive Methods ---
    {'name': 'Adam', 'opt': 'Adam', 'lr': 0.001, 'wd': 0.0, 'act': 'relu', 'drop': 0.0, 'bn': False},
    {'name': 'RMSprop', 'opt': 'RMSprop', 'lr': 0.001, 'wd': 0.0, 'act': 'relu', 'drop': 0.0, 'bn': False},
    
    # --- Regularization & Architecture Variants ---
    {'name': 'Adam + BatchNorm', 'opt': 'Adam', 'lr': 0.001, 'wd': 0.0, 'act': 'relu', 'drop': 0.0, 'bn': True},
    {'name': 'SGD + Mom + BatchNorm', 'opt': 'SGD_Momentum', 'lr': 0.001, 'wd': 0.0, 'act': 'relu', 'drop': 0.0, 'bn': True},
    {'name': 'Adam + Dropout', 'opt': 'Adam', 'lr': 0.001, 'wd': 0.0, 'act': 'relu', 'drop': 0.2, 'bn': False},
    {'name': 'Adam + L2 (WD)', 'opt': 'Adam', 'lr': 0.001, 'wd': 0.01, 'act': 'relu', 'drop': 0.0, 'bn': False},
    
    # --- Activation Variants ---
    {'name': 'Adam + LeakyReLU', 'opt': 'Adam', 'lr': 0.001, 'wd': 0.0, 'act': 'leaky_relu', 'drop': 0.0, 'bn': False},
    {'name': 'Adam + Tanh', 'opt': 'Adam', 'lr': 0.001, 'wd': 0.0, 'act': 'tanh', 'drop': 0.0, 'bn': False}
]

results = {}
epochs = 100

print(f"Training with {epochs} epochs...")

for conf in configs:
    print(f"Running {conf['name']}...")
    losses, duration = train_model(
        conf['opt'], 
        learning_rate=conf['lr'], 
        epochs=epochs, 
        weight_decay=conf['wd'],
        activation=conf['act'],
        dropout=conf['drop'],
        use_batchnorm=conf['bn']
    )
    results[conf['name']] = {'loss': losses, 'time': duration}
    print(f"  -> Time: {duration:.4f}s, Final Loss: {losses[-1]:.4f}")

In [ ]:
results = {}
epochs = 100

print(f"Training with {epochs} epochs...")

for conf in configs:
    print(f"Running {conf['name']}...")
    losses, duration = train_model(
        conf['opt'], 
        learning_rate=conf['lr'], 
        epochs=epochs, 
        weight_decay=conf['wd'],
        activation=conf['act'],
        dropout=conf['drop']
    )
    results[conf['name']] = {'loss': losses, 'time': duration}
    print(f"  -> Time: {duration:.4f}s, Final Loss: {losses[-1]:.4f}")

In [ ]:
# Gradient Descent Variants Comparison

results = {}
epochs = 100

print(f"Training with {epochs} epochs...")

for conf in configs:
    print(f"Running {conf['name']}...")
    losses, duration = train_model(
        conf['opt'], 
        learning_rate=conf['lr'], 
        epochs=epochs, 
        weight_decay=conf['wd'],
        activation=conf['act'],
        dropout=conf['drop']
    )
    results[conf['name']] = {'loss': losses, 'time': duration}
    print(f"  -> Time: {duration:.4f}s, Final Loss: {losses[-1]:.4f}")

## 5. Visualization and Comparison

In [ ]:
# Plot Loss Curves
plt.figure(figsize=(14, 8))
for name, data in results.items():
    plt.plot(data['loss'], label=name, linewidth=1.5)

plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.title('Training Loss Convergence by Configuration')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Plot Training Time
names = list(results.keys())
times = [results[n]['time'] for n in names]

plt.figure(figsize=(12, 6))
plt.bar(names, times, color='skyblue')
plt.ylabel('Time (seconds)')
plt.title('Total Training Time (100 Epochs)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 6. Deep Dive: Standardization vs. Batch Normalization

### Why do we need them?
Neural networks train best when inputs are on a similar scale (e.g., 0 to 1 or mean 0, variance 1). If one feature has a range of [0, 1000] and another [0, 1], the gradients for the weights connected to the first feature will be huge, while the others will be tiny. This makes the loss landscape "narrow" and hard to navigate, forcing us to use very small learning rates.

### 1. Standard Scaler (Input Normalization)
- **What it does**: Transforms the **input data** ($X$) to have mean 0 and standard deviation 1.
- **Where applied**: Only on the raw input features before feeding them into the network.
- **Benefit**: Ensures all input features contribute equally to the initial learning. Prevents gradients from exploding early on.
- **Disadvantage**: It relies on the statistics (mean/std) of the training set. If the test set has a very different distribution (domain shift), the model may perform poorly.

### 2. Batch Normalization (Layer Normalization)
- **What it does**: Normalizes the **activations** of hidden layers *during* training. It calculates the mean and variance of the current *mini-batch* and normalizes the layer's output.
- **Where applied**: Inside the network, typically after a Linear/Conv layer and before the Activation function.
- **Benefit**:
    - **Stabilizes Training**: Prevents "Internal Covariate Shift" (where the distribution of layer inputs keeps changing as previous layers update).
    - **Higher Learning Rates**: Allows using much larger learning rates without diverging.
    - **Regularization**: Adds slight noise (due to batch statistics), reducing overfitting.
- **Disadvantage**:
    - **Computational Cost**: Adds extra calculations per step.
    - **Batch Size Dependency**: Doesn't work well with very small batch sizes (e.g., < 4) because the batch statistics become too noisy.

### Impact on Data
- **Loss of Information?**: No. Both techniques are linear transformations ($x' = \frac{x - \mu}{\sigma}$). They shift and scale the data but preserve the relative relationships and distances between points. BatchNorm even learns a scale ($\gamma$) and shift ($\beta$) parameter to undo the normalization if the network decides the original distribution was better.

### When to use?
- **Standard Scaler**: Almost **ALWAYS** for tabular data or continuous features in Neural Networks.
- **Batch Norm**: Highly recommended for **Deep Networks** (many layers) or when training is unstable/slow. Less critical for very shallow networks but rarely hurts.

## 7. Observations

1.  **Convergence Speed**:?

2.  **Training Time**:?